## Manufacturing Case
Providing context-sensitive decision-support for PID parameter tuning

### 0. Import functions shared with the other domain examples

In [1]:
import sys
sys.path.append("..")
from yarrrmlHandler import YarrrmlHandler
from etc import export_sheets_to_csv
from rdf_etc import run_query_file, add_namespaces, show_rdf_graph

### 1. Export all Excel worksheets as csv files (YARRRML does not inherently support xlsx files)

In [2]:
for filename in ['claimGenerationNetwork', 'claimFormalizationNetwork']:
    export_sheets_to_csv(
        excel_file=filename + '.xlsx', 
        output_dir='csv'
)

Exported experts -> csv\experts.csv
Exported events -> csv\events.csv
Exported resources -> csv\resources.csv
Exported claims -> csv\claims.csv
Exported claims -> csv\claims.csv
Exported kEngineers -> csv\kEngineers.csv
Exported formalRepresentations -> csv\formalRepresentations.csv
Exported assessments -> csv\assessments.csv
Exported CQs -> csv\CQs.csv
Exported formalCQs -> csv\formalCQs.csv


### 2. Convert csv data to RDF representation using the YARRRML specification
The graph is visualized using [yfiles-graphs-for-jupter](https://www.yworks.com/products/yfiles-graphs-for-jupyter). Please refer to the official documentation for using the exploration features. The visualization is designed for networkx graphs, not RDF graphs in specific. If you want to explore the graph with a specialized tool, please export the graph by uncommenting the serialization contextmanager.

In [3]:
from rdflib import Graph

try:
    result = YarrrmlHandler.run_pipeline(
        working_dir=".",
        yarrrml_file="manufacturing.yarrrml.yml",
        rml_file="manufacturing_rules.rml.ttl",
    )
    print("Pipeline executed successfully.")
except Exception as ex:
    print(f"Error: {ex}")

g =  Graph().parse(data=result.stdout, format="nt")
g = add_namespaces(g)

# Uncomment to export RDF graph
# with open("rdf_export.ttl", mode="w", encoding="utf8") as f:
#     f.write(g.serialize(format="turtle"))
    
show_rdf_graph(g)

Pipeline executed successfully.


GraphWidget(layout=Layout(height='800px', width='100%'))

### 3. Store conditional knowledge base as annotation page as snapshot for further processing
This step is skipped for this demonstration as we can only compile a knowledge base subset from the RDF graph. Please refer to one of the other example for this step.

### 4. Extract rules from selected cases and load knowledge base to interactive consultant for decision-support.
1. Select the cases which are deemed similar to the current case where we want to provide decision-support. In practice, the context evaluation is automated, but for the purpose of demonstration, the reader can select the cases from which related rules should be added to the knowledge base.
2. Select a conflict resolution mechanism:
    * _Consensus_: Conflicting rules are automatically resolved by choosing the rule which is most represented among the cases. An equal amount will raise an error and cannot be resolved automatically.
    *  _Order_: Conflicting rules are automatically resolved by choosing the rule which occurres first in the priority order. If conflicting rules are documented within the same case, an error is raised because the conflict cannot be resolved automatically.
    *  This step should highlight how the architecture supports knowledge engineers in supporting conflict resolvement either by identifying candidates or automatic resolvement.
3. After pressing the button "Generate KB", a new browser tab is opened displaying IDP-Z3's interactive consultant with the loaded knowledge base. The goal is to propose which tuning parameters should be changed by what magnitude in widget "tune" under consideration of the selected values of the other widgets. _Remark: The GUI below is used to only infer the applicable positive and negative effect rules of the knowledge base. The exact output is printed below the button after execution. The output is injected in the [KB template](kb_template.idp) and then send to the interactive consultant.

In [4]:
from functools import partial
import ipywidgets as widgets

from gui import create_ui, add_cases, remove_case, move_up, move_down, generate_kb

results = run_query_file(g, "get_all_cases.rq")
case_ids = sorted({str(row[0]) for row in results})

# Create UI Elements
(
    ui,
    available,
    selected,
    conflict_resolution,
    add_btn,
    remove_btn,
    up_btn,
    down_btn,
    generate_btn,
) = create_ui(case_ids)

output = widgets.Output()

# Add EventListeners
add_btn.on_click(partial(add_cases, available=available, selected=selected))
remove_btn.on_click(partial(remove_case, selected=selected))
up_btn.on_click(partial(move_up, selected=selected))
down_btn.on_click(partial(move_down, selected=selected))
generate_btn.on_click(partial(generate_kb, graph=g, selected=selected, conflict_resolution=conflict_resolution, output=output))

display(ui)
display(output)

Output()